# 📚 Conversational RAG: Question Answering with Chat History

## Learning Objectives
In this notebook, you will learn:
1. **A retrieval chain** - answer questions over a blog post with `create_retrieval_chain`
2. **Why follow-ups fail** - see a retriever search for "it" without knowing what "it" is
3. **History-aware retrieval** - rewrite each follow-up into a standalone question before retrieving
4. **Automatic history** - wrap the conversational RAG chain in `RunnableWithMessageHistory`

## Where this fits
The earlier notebooks sent history to the *model*. In retrieval-augmented generation the *retriever* needs it
too: a follow-up like "How do we achieve it?" retrieves nothing useful on its own. This notebook applies the
memory patterns from `03_runnable_with_message_history.ipynb` to a RAG chain.

## Prerequisites
- `03_runnable_with_message_history.ipynb`; basic RAG (loaders, splitters, vector stores)
- Dependencies installed with `uv sync` at the repository root
- `OPENAI_API_KEY` in the project-root `.env` (chat model and embeddings)
- Internet access: the notebook downloads one public blog post

> **⚠️ Legacy API**: `create_retrieval_chain`, `create_stuff_documents_chain` and `create_history_aware_retriever` are legacy chain constructors kept in `langchain_classic`; `WebBaseLoader` comes from the sunset `langchain-community`; `RunnableWithMessageHistory` (Part 4) is deprecated in `langchain-core` 1.3.3 and removed in 2.0. LangChain now recommends LangGraph persistence instead — a checkpointer
> and a `thread_id` — taught in [`01_memory_and_conversational_agent.ipynb`](06a_memory_and_conversational_agent.ipynb)
> (module 02). This module keeps the older API because existing LCEL code uses it everywhere; the setup cell
> lets its deprecation warnings through so you see them where the deprecated classes are used.


---
## 📦 Step 1: Environment Setup

The setup cell loads the API keys, sets a descriptive `USER_AGENT` for the web loader and silences library
warnings — except LangChain's deprecation warnings, which it lets through because this notebook uses deprecated
APIs on purpose.

In [ ]:
# ============================================================================
# ENVIRONMENT SETUP: Load API keys and suppress warnings
# ============================================================================
import os
import warnings

from dotenv import find_dotenv, load_dotenv
from langchain_core._api import LangChainDeprecationWarning

warnings.filterwarnings("ignore")
# This notebook uses deprecated LangChain APIs on purpose: let their deprecation warnings through
warnings.filterwarnings("default", category=LangChainDeprecationWarning)

# A descriptive User-Agent for the web loader in Part 1
os.environ.setdefault("USER_AGENT", "aied-08-conversational-rag-notebook")

# Reads OPENAI_API_KEY from the project-root .env
load_dotenv(find_dotenv(usecwd=True))

print("✅ Environment ready")

In [ ]:
# ============================================================================
# LLM INITIALIZATION: answers questions and rewrites follow-ups
# ============================================================================
from aied_helpers import get_openai_llm

llm = get_openai_llm(model="gpt-4o-mini")

print(f"✅ LLM initialized: {llm.model_name}")

---
## 📄 Part 1: A Retrieval Chain over a Blog Post

We index Lilian Weng's post on LLM-powered agents: load → split into 1000-character chunks → embed into an
in-memory Chroma collection → expose as a retriever.

In [ ]:
# ============================================================================
# PART 1A: LOAD, SPLIT AND INDEX THE POST
# ============================================================================
import bs4
from aied_helpers import get_embeddings
from langchain_chroma import Chroma
from langchain_community.document_loaders import WebBaseLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

loader = WebBaseLoader(
    web_paths=("https://lilianweng.github.io/posts/2023-06-23-agent/",),
    bs_kwargs=dict(parse_only=bs4.SoupStrainer(class_=("post-content", "post-title", "post-header"))),
)
docs = loader.load()

splits = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200).split_documents(docs)
vectorstore = Chroma.from_documents(documents=splits, embedding=get_embeddings(provider="openai", model="text-embedding-3-small"))
retriever = vectorstore.as_retriever()

print(f"Loaded {len(docs)} document ({len(docs[0].page_content):,} characters) -> {len(splits)} chunks indexed")

In [ ]:
# ============================================================================
# PART 1B: A STUFF-DOCUMENTS QA CHAIN
# ============================================================================
from langchain_classic.chains import create_retrieval_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate

system_prompt = (
    "You are an assistant for question-answering tasks. Use the following pieces of retrieved context to "
    "answer the question. If you don't know the answer, say that you don't know. Use three sentences maximum "
    "and keep the answer concise.\n\n{context}"
)
qa_prompt = ChatPromptTemplate.from_messages([("system", system_prompt), ("human", "{input}")])

rag_chain = create_retrieval_chain(retriever, create_stuff_documents_chain(llm, qa_prompt))

response = rag_chain.invoke({"input": "What is Self-Reflection?"})
print(response["answer"])

---
## ❓ Part 2: Follow-up Questions Without History

The retriever receives the literal text of the question. For a follow-up, that text has no subject.

In [ ]:
# ============================================================================
# PART 2: A FOLLOW-UP THE RETRIEVER CANNOT RESOLVE
# ============================================================================
follow_up = rag_chain.invoke({"input": "How do we achieve it?"})

print("Chunks retrieved for 'How do we achieve it?':")
for doc in follow_up["context"]:
    print("  -", doc.page_content[:90].replace("\n", " "), "…")
print(f"\nAnswer: {follow_up['answer']}")

Look at the retrieved chunks: they were chosen by similarity to "How do we achieve it?", not to
self-reflection, so the answer is either vague or about something else.

---
## 🧭 Part 3: History-Aware Retrieval

`create_history_aware_retriever` adds one step before retrieval: an LLM rewrites the latest question into a
**standalone question** using the chat history, and that rewritten question is what the retriever searches for.
The answering prompt also receives the history.

In [ ]:
# ============================================================================
# PART 3A: THE QUESTION-REWRITING PROMPT
# ============================================================================
from langchain_classic.chains import create_history_aware_retriever
from langchain_core.messages import AIMessage, HumanMessage
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import MessagesPlaceholder

contextualize_q_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "Given a chat history and the latest user question which might reference context in the chat history, "
     "formulate a standalone question which can be understood without the chat history. Do NOT answer the "
     "question, just reformulate it if needed and otherwise return it as is."),
    MessagesPlaceholder("chat_history"),
    ("human", "{input}"),
])

# See the rewrite on its own before wiring it in
demo_history = [HumanMessage(content="What is Self-Reflection?"), AIMessage(content=response["answer"])]
rewrite = (contextualize_q_prompt | llm | StrOutputParser()).invoke({"input": "How do we achieve it?", "chat_history": demo_history})
print(f"'How do we achieve it?'  ->  {rewrite!r}")

In [ ]:
# ============================================================================
# PART 3B: THE CONVERSATIONAL RAG CHAIN
# ============================================================================
history_aware_retriever = create_history_aware_retriever(llm, retriever, contextualize_q_prompt)

conversational_qa_prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    MessagesPlaceholder("chat_history"),
    ("human", "{input}"),
])
conversational_rag_chain = create_retrieval_chain(
    history_aware_retriever, create_stuff_documents_chain(llm, conversational_qa_prompt)
)

# Manage the history by hand for two turns
chat_history = []
question = "What is Self-Reflection?"
first = conversational_rag_chain.invoke({"input": question, "chat_history": chat_history})
chat_history += [HumanMessage(content=question), AIMessage(content=first["answer"])]

question2 = "How do we achieve it?"
second = conversational_rag_chain.invoke({"input": question2, "chat_history": chat_history})
print("Chunks retrieved now:")
for doc in second["context"]:
    print("  -", doc.page_content[:90].replace("\n", " "), "…")
print(f"\nAnswer: {second['answer']}")

Compare the retrieved chunks with Part 2: the retriever searched for the rewritten, self-reflection-specific
question, so the chunks and the answer are now about how self-reflection is achieved.

---
## 🔄 Part 4: Automatic History with RunnableWithMessageHistory

The retrieval chain returns a dict (`input`, `context`, `answer`), so the wrapper needs one more key:
**`output_messages_key="answer"`** tells it which output to save as the AI message.

In [ ]:
# ============================================================================
# PART 4: CONVERSATIONAL RAG WITH SESSION HISTORY
# ============================================================================
from langchain_core.chat_history import BaseChatMessageHistory, InMemoryChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory

store: dict[str, InMemoryChatMessageHistory] = {}


def get_session_history(session_id: str) -> BaseChatMessageHistory:
    if session_id not in store:
        store[session_id] = InMemoryChatMessageHistory()
    return store[session_id]


rag_with_history = RunnableWithMessageHistory(
    conversational_rag_chain,
    get_session_history,
    input_messages_key="input",
    history_messages_key="chat_history",
    output_messages_key="answer",
)

session = {"configurable": {"session_id": "abc123"}}
for q in ["What is Task Decomposition?", "What are common ways of doing it?"]:
    print(f"User: {q}\nAI:   {rag_with_history.invoke({'input': q}, config=session)['answer']}\n")

print(f"Messages stored for abc123: {len(store['abc123'].messages)}")

---
## 📝 Summary

In this notebook, we learned:

### 1. A retrieval chain
- `create_retrieval_chain(retriever, create_stuff_documents_chain(llm, prompt))` answers from retrieved chunks

### 2. Why follow-ups fail
- Without history the retriever searches for the literal follow-up text and finds the wrong chunks

### 3. History-aware retrieval
- `create_history_aware_retriever` rewrites each question into a standalone one before retrieving

### 4. Automatic history
- `RunnableWithMessageHistory(..., output_messages_key="answer")` stores the answer from a dict output

### Next Steps
- **`07_multi_user_product_recommender.ipynb`**: combine several chains with per-user memory in an application